In [18]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Literal, Optional
from langchain_core.messages import BaseMessage, AIMessage, HumanMessage, SystemMessage
from langgraph.graph.message import add_messages  ## Optimized reducer with BaseMessage Class
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
from my_llm import *
from pydantic import BaseModel, Field
from langchain_core.runnables import RunnableConfig
from langchain.tools import tool
from langgraph.prebuilt import ToolNode, tools_condition 
import sqlite3
from tools import calculate, search_tool, get_stock_price

from datetime import datetime



In [ ]:
class Router(BaseModel):
    mode: Literal["model_only","hybrid","search_required"] = Field(description="Depending on the topic, decide whether internet search is required fully or partly or not at all")
    queries: Optional[list[str]] = Field(description="In case internet search is required, what queries to search for")
    reasoning: Optional[str] = Field(description="Reasoning behind the decision of search category and queries")
    needs_research: bool = Field(description="Whether internet search is required for the task or not")

In [ ]:
class State(TypedDict):
    topic: str
    internet_search_required: Optional[bool] = Field(description="Whether internet search is required for the task or not")
    reasoning: Optional[str] = Field(description="Reasoning behind the decision of search category and queries")
    search_category: Literal["model_only","hybrid","search_required"] = Field(description="Depending on the topic, decide whether internet search is required fully or partly or not at all")
    queries: Optional[list[str]] = Field(description="In case internet search is required, what queries to search for")

In [67]:
def router_fn(state: State) -> Router:
    ## This is a dummy router function, you can replace it with your own implementation which can be more complex and can use LLMs to make the routing decision based on the topic and other factors.
    message = [
        SystemMessage(content="Decide the search category and queries based on the topic. Note that the current date today is " + datetime.now().strftime("%Y-%m-%d")),
        HumanMessage(content=f"Topic: {state['topic']}")
    ]
    response = model.with_structured_output(Router).invoke(message)
    return {"search_category": response.search_category, "queries": response.queries, "internet_search_required": response.internet_search_required, "reasoning": response.reasoning}

In [68]:
graph = StateGraph(State)
graph.add_node("Router", router_fn)

graph.set_entry_point("Router")
graph.set_finish_point("Router")

workflow = graph.compile()

In [69]:
workflow.invoke({"topic": "Latest advancements in AI from 2022 to 2026"})

c:\Users\shivo\Projects\Agentic\Agentic_Learning\langenv\lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Router(search_category='s...et_search_required=True), input_type=Router])
  return self.__pydantic_serializer__.to_python(


{'topic': 'Latest advancements in AI from 2022 to 2026',
 'internet_search_required': True,
 'reasoning': 'The topic is focused on the most recent and specific time range advancements in AI. To ensure accurate and up-to-date information on the latest breakthroughs and trends within this period, an internet search is necessary.',
 'search_category': 'search_required',
 'queries': ['Latest advancements in AI from 2022 to 2026',
  'Breakthrough AI developments 2022-2026',
  'Recent trends in artificial intelligence advancements 2022-2026']}

In [1]:
from langchain_tavily import TavilySearch
import os, getpass
from dotenv import load_dotenv

load_dotenv()

TAVILY_API_KEY =  os.getenv("TAVILY_API_KEY")
os.environ["TAVILY_API_KEY"] = TAVILY_API_KEY

In [6]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
    max_results=5,
    topic="general",
)

In [8]:
search_tool.invoke("Latest advancements in AI from 2022 to 2026")

{'query': 'Latest advancements in AI from 2022 to 2026',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://hqsoftwarelab.com/blog/latest-ai-developments/',
   'title': 'Recent AI Developments in 2026: Latest AI Trends - HQSoftware',
   'content': 'Breakthroughs in Machine Learning (ML), AI-driven automation, and natural language processing make the field of AI more dynamic and promising',
   'score': 0.78493005,
   'raw_content': None},
  {'url': 'https://www.trigyn.com/insights/ai-trends-2026-new-era-ai-advancements-and-breakthroughs',
   'title': 'AI Trends in 2026: A New Era of AI Advancements and Breakthroughs',
   'content': 'Explore top AI trends for 2026, from enterprise adoption to autonomous systems and breakthrough business innovations.',
   'score': 0.77096564,
   'raw_content': None},
  {'url': 'https://sloanreview.mit.edu/article/five-trends-in-ai-and-data-science-for-2026/',
   'title': 'Five Trends in AI and Data Science for 2026

In [15]:
# Basic usage
result = tool.invoke({"query": "Virat kohli performance in IPL 2025"})

In [16]:
from langchain.agents import create_agent

In [ ]:
# Initialize the agent with the search tool
agent = create_agent(
    model=model,
    tools=[search_tool],
    system_prompt="You are a helpful research assistant. Use web search to find accurate, up-to-date information."
)

# Use the agent
response = agent.invoke({
    "messages": [{"role": "user", "content": "What is the latest news about justin bieber"}]
})

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Evidence_item(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = Field(None, description="When was the article published? If available")
    snippet: Optional[str] = Field(None, description="A brief snippet from the article that is relevant to the task")
    source: Optional[str] = Field(None, description="Source of the evidence, e.g., website name or database name")

class EvidencePackage(BaseModel):
    evidence_items: List[Evidence_item]

def ResearchAgent():
    search_tool = TavilySearch(
        max_results=2,
        topic="general",
    )

    evidence_items = []

    queries = [
        "who is Narendra Modi?",
        "Why is Football the most popular sport in the world?"
    ]

    for query in queries:
        results = search_tool.invoke(query)

        for item in results.get("results", []):
            evidence = Evidence_item(
                title=item.get("title", ""),
                url=item.get("url", ""),
                published_at=item.get("published_date"),
                snippet=item.get("content"),
                source=item.get("url", "").split("/")[2] if item.get("url") else None
            )
            evidence_items.append(evidence)

    # ✅ Wrap into EvidencePackage
    return EvidencePackage(evidence_items=evidence_items)

In [16]:
ResearchAgent()

[Evidence_item(title='Narendra Modi | Biography, Career, & Facts', url='https://www.britannica.com/biography/Narendra-Modi', published_at=None, snippet='[References & Edit History](https://www.britannica.com/biography/Narendra-Modi/additional-info)  [Quick Facts & Related Topics](/facts/Narendra-Modi). Narendra Modi served as the [chief minister](https://www.britannica.com/topic/chief-minister-India) of [Gujarat](https://www.britannica.com/place/Gujarat) from 2001 to 2014. Narendra Modi was first elected [prime minister of India](https://www.britannica.com/topic/Prime-Minister-of-India) in 2014. **Narendra Modi** (born September 17, 1950, Vadnagar, India) is an Indian politician and government official who rose to become a senior leader of the [Bharatiya Janata Party](https://www.britannica.com/topic/Bharatiya-Janata-Party) (BJP). The Modi-led BJP (and the BJP-led [NDA](https://www.britannica.com/topic/National-Democratic-Alliance-political-organization-India) alliance) won the majorit